# CNN + BiLSTM + Attention — SEED-IV (Trial-Grouped Split, Per-Subject Normalization)
Loads directly from your cleaned CSV. Groups consecutive samples within each trial into sliding windows so the BiLSTM sees a real temporal sequence.

Changes vs. the previous version:
- **Per-subject normalization**: EEG baselines vary a lot between people. Z-scoring each subject's features using *that subject's own* mean/std removes most of this inter-subject offset, so the model can't shortcut by learning subject identity instead of emotion-relevant patterns. (This replaces the single global `StandardScaler`.)
- **Smaller model** (`cnn_ch=128`, `lstm_h=64`) — the previous 256/128 config memorized the training trials very quickly (98% train acc by epoch 25) while validation stayed around 45-50%.
- **Stronger regularization**: `DROPOUT=0.6`, `WEIGHT_DECAY=1e-3`, lower `LR=5e-4`.

Trial groups still kept out of both train and test via `GroupShuffleSplit`.

## 1. Imports & Config

In [1]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import accuracy_score, f1_score, classification_report



In [2]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

# ── Hyperparameters ───────────────────────────────────────────
WINDOW_SIZE  = 10      # consecutive samples per sequence fed to BiLSTM
STEP_SIZE    = 5       # sliding window stride
BATCH_SIZE   = 64
EPOCHS       = 80
LR           = 5e-4    # lowered from 1e-3 — training acc was climbing too fast
DROPOUT      = 0.6     # raised from 0.5
WEIGHT_DECAY = 1e-3     # raised from 1e-4
PATIENCE     = 15      # early stopping

Device: cpu


In [3]:
# ── Load CSV ─────────────────────────────────────────────────
df = pd.read_csv("../data/neurosense_cleaned.csv")
print(f'Loaded: {df.shape}  |  Label distribution:\n{df["label"].value_counts().sort_index()}')

# ── Feature columns ────────────────────────────────────────────
EEG_COLS  = [f'eeg_feature_{i}' for i in range(1, 311)]   # 310 features
EYE_COLS  = [f'eye_feature_{i}' for i in range(1, 32)]    # 31 features
STAT_COLS = ['avg_eeg','std_eeg','max_eeg','min_eeg',
             'avg_eye','std_eye','max_eye','min_eye',
             'eeg_eye_ratio','eeg_range','eye_range',
             'eeg_stability','eye_stability','phys_activity']
FEAT_COLS = EEG_COLS + EYE_COLS + STAT_COLS   # 355 total
META_COLS = ['subject', 'session', 'trial']

INPUT_SIZE  = len(FEAT_COLS)   # 355
NUM_CLASSES = df['label'].nunique()
print(f'Features: {INPUT_SIZE}  |  Classes: {NUM_CLASSES}  |  Subjects: {df["subject"].nunique()}')


df[FEAT_COLS] = (
    df.groupby('subject')[FEAT_COLS]
      .transform(lambda x: (x - x.mean()) / (x.std() + 1e-8))
)

print('Per-subject normalization applied.')
print(df[FEAT_COLS].describe().loc[['mean', 'std']].iloc[:, :4])

Loaded: (37575, 361)  |  Label distribution:
label
0    9750
1    9525
2    9735
3    8565
Name: count, dtype: int64
Features: 355  |  Classes: 4  |  Subjects: 15
Per-subject normalization applied.
      eeg_feature_1  eeg_feature_2  eeg_feature_3  eeg_feature_4
mean  -8.048090e-16   1.422031e-15   5.567100e-16   9.681913e-16
std    9.998137e-01   9.998137e-01   9.998137e-01   9.998137e-01


In [4]:
def build_windows(df, feat_cols, window_size, step_size):
    X_list = []
    y_list = []
    g_list = []

    grouped = df.groupby(["subject", "session", "trial"], sort=False)

    for group_id, (_, grp) in enumerate(grouped):
        feats = grp[feat_cols].values.astype(np.float32)
        label = int(grp["label"].iloc[0])
        T = len(feats)

        if T < window_size:
            pad = np.repeat(feats[-1:], window_size - T, axis=0)
            feats = np.vstack([feats, pad])
            T = window_size

        for start in range(0, T - window_size + 1, step_size):
            X_list.append(feats[start:start + window_size])
            y_list.append(label)
            g_list.append(group_id)

    return (
        np.array(X_list, dtype=np.float32),
        np.array(y_list, dtype=np.int64),
        np.array(g_list, dtype=np.int64)
    )


X, y, groups = build_windows(df, FEAT_COLS, WINDOW_SIZE, STEP_SIZE)

print(f"Windows X: {X.shape}")
print(f"Labels y: {y.shape}")
print(f"Groups: {groups.shape}")
print(f"Number of trial groups: {len(np.unique(groups))}")





Windows X: (5955, 10, 355)
Labels y: (5955,)
Groups: (5955,)
Number of trial groups: 1080


In [5]:
# ── Train/Test split by trial group ────────────────────────────
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.2,
    random_state=SEED
)

train_idx, test_idx = next(gss.split(X, y, groups))

X_train, X_test = X[train_idx], X[test_idx]
y_train, y_test = y[train_idx], y[test_idx]

train_groups = set(groups[train_idx])
test_groups = set(groups[test_idx])

assert len(train_groups & test_groups) == 0, "Trial leakage detected!"

print(f"Train: {X_train.shape}")
print(f"Test: {X_test.shape}")
print(f"Train trials: {len(train_groups)}")
print(f"Test trials: {len(test_groups)}")

def make_loader(X, y, batch_size, shuffle):
    dataset = TensorDataset(
        torch.from_numpy(X),
        torch.from_numpy(y)
    )

    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        pin_memory=True
    )


train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
test_loader = make_loader(X_test, y_test, BATCH_SIZE, shuffle=False)

Train: (4820, 10, 355)
Test: (1135, 10, 355)
Train trials: 864
Test trials: 216


In [6]:
class Attention(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.w = nn.Linear(dim, 1, bias=False)

    def forward(self, x):
        weights = torch.softmax(self.w(x).squeeze(-1), dim=1)
        return (weights.unsqueeze(1) @ x).squeeze(1)


class CNN_BiLSTM(nn.Module):
    def __init__(
        self,
        input_size,
        cnn_ch=128,
        lstm_h=64,
        lstm_layers=2,
        num_classes=4,
        dropout=0.6
    ):
        super().__init__()

        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, cnn_ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_ch),
            nn.GELU(),
            nn.Dropout(dropout * 0.4),

            nn.Conv1d(cnn_ch, cnn_ch, kernel_size=3, padding=1),
            nn.BatchNorm1d(cnn_ch),
            nn.GELU(),
            nn.Dropout(dropout * 0.4),
        )

        self.bilstm = nn.LSTM(
            input_size=cnn_ch,
            hidden_size=lstm_h,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        self.attn = Attention(lstm_h * 2)

        self.head = nn.Sequential(
            nn.Linear(lstm_h * 2, 128),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.cnn(x)

        x = x.permute(0, 2, 1)
        x, _ = self.bilstm(x)

        x = self.attn(x)
        return self.head(x)


In [7]:
# ── Two architectures to compare ──────────────────────────────
architectures = [
    {
        "name": "CNN-BiLSTM-Attention A",
        "cnn_ch": 128,
        "lstm_h": 64,
        "lstm_layers": 2,
        "dropout": 0.6
    },
    {
        "name": "CNN-BiLSTM-Attention B",
        "cnn_ch": 64,
        "lstm_h": 128,
        "lstm_layers": 1,
        "dropout": 0.5
    }
]


emotions = {
    0: "Neutral",
    1: "Sad",
    2: "Fear",
    3: "Happy"
}

results = []

best_overall_acc = 0.0
best_overall_state = None
best_overall_name = None



In [8]:
# ── Training loop for both architectures ──────────────────────
for arch in architectures:
    print("\n" + "=" * 70)
    print(f"Training architecture: {arch['name']}")
    print("=" * 70)

    model = CNN_BiLSTM(
        input_size=INPUT_SIZE,
        cnn_ch=arch["cnn_ch"],
        lstm_h=arch["lstm_h"],
        lstm_layers=arch["lstm_layers"],
        num_classes=NUM_CLASSES,
        dropout=arch["dropout"]
    ).to(DEVICE)

    n_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    print(f"Trainable parameters: {n_params:,}")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

    optimizer = optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=EPOCHS,
        eta_min=1e-5
    )

    best_acc = 0.0
    best_state = None
    no_improve = 0

    print(f'{"Ep":>4}  {"Tr Loss":>8}  {"Tr Acc":>7}  {"Val Acc":>7}')
    print("-" * 35)

    for ep in range(1, EPOCHS + 1):

        model.train()

        tr_loss = 0
        tr_correct = 0
        tr_total = 0

        for xb, yb in train_loader:
            xb = xb.to(DEVICE)
            yb = yb.to(DEVICE)

            optimizer.zero_grad()

            logits = model(xb)
            loss = criterion(logits, yb)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()

            tr_loss += loss.item() * xb.size(0)
            tr_correct += (logits.argmax(1) == yb).sum().item()
            tr_total += xb.size(0)

        scheduler.step()

        model.eval()

        preds = []
        labels = []

        with torch.no_grad():
            for xb, yb in test_loader:
                xb = xb.to(DEVICE)

                logits = model(xb)
                pred = logits.argmax(1).cpu().numpy()

                preds.extend(pred)
                labels.extend(yb.numpy())

        val_acc = accuracy_score(labels, preds)

        if val_acc > best_acc:
            best_acc = val_acc
            best_state = {
                k: v.cpu().clone()
                for k, v in model.state_dict().items()
            }
            no_improve = 0
        else:
            no_improve += 1

        if ep % 5 == 0 or ep == 1:
            print(
                f"{ep:>4}  {tr_loss / tr_total:>8.4f}  "
                f"{tr_correct / tr_total * 100:>6.2f}%  "
                f"{val_acc * 100:>6.2f}%"
            )

        if no_improve >= PATIENCE:
            print(
                f"Early stop at epoch {ep} | "
                f"Best val acc: {best_acc * 100:.2f}%"
            )
            break
     # ── Final evaluation ──────────────────────────────────────
    model.load_state_dict(best_state)
    model.eval().to(DEVICE)

    preds = []
    labels = []

    with torch.no_grad():
        for xb, yb in test_loader:
            xb = xb.to(DEVICE)

            logits = model(xb)
            pred = logits.argmax(1).cpu().numpy()

            preds.extend(pred)
            labels.extend(yb.numpy())

    acc = accuracy_score(labels, preds)
    f1 = f1_score(labels, preds, average="weighted")

    print("\nFinal results:")
    print(f"Architecture : {arch['name']}")
    print(f"Test Accuracy: {acc * 100:.2f}%")
    print(f"Weighted F1  : {f1:.4f}")
    print()

    print(
        classification_report(
            labels,
            preds,
            target_names=[emotions[i] for i in range(NUM_CLASSES)]
        )
    )

    results.append({
        "Model": arch["name"],
        "cnn_ch": arch["cnn_ch"],
        "lstm_h": arch["lstm_h"],
        "lstm_layers": arch["lstm_layers"],
        "dropout": arch["dropout"],
        "accuracy": acc,
        "weighted_f1": f1,
        "trainable_params": n_params
    })

    if acc > best_overall_acc:
        best_overall_acc = acc
        best_overall_state = best_state
        best_overall_name = arch["name"]


Training architecture: CNN-BiLSTM-Attention A
Trainable parameters: 402,052
  Ep   Tr Loss   Tr Acc  Val Acc
-----------------------------------


c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   1    1.2816   40.79%   43.17%


c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


   5    0.5502   90.79%   49.25%


...

c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\L

  25    0.3823   98.98%   49.52%


c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Early stop at epoch 29 | Best val acc: 50.66%

Final results:
Architecture : CNN-BiLSTM-Attention B
Test Accuracy: 50.66%
Weighted F1  : 0.5008

              precision    recall  f1-score   support

     Neutral       0.56      0.59      0.57       307
         Sad       0.46      0.33      0.39       302
        Fear       0.48      0.60      0.53       278
       Happy       0.52      0.51      0.52       248

    accuracy                           0.51      1135
   macro avg       0.50      0.51      0.50      1135
weighted avg       0.50      0.51      0.50      1135



c:\Users\Lenovo\anaconda3\envs\dropout\Lib\site-packages\torch\utils\data\dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [9]:
# ── Comparison table ──────────────────────────────────────────
results_df = pd.DataFrame(results)

print("\nArchitecture Comparison:")
print(results_df)


# ── Save results and best model ───────────────────────────────
results_df.to_csv("cnn_bilstm_architecture_comparison.csv", index=False)

torch.save(
    best_overall_state,
    "cnn_bilstm_seed_iv_best_architecture.pth"
)

print(f"\nBest architecture: {best_overall_name}")
print(f"Best Accuracy: {best_overall_acc * 100:.2f}%")
print("Saved model: cnn_bilstm_seed_iv_best_architecture.pth")
print("Saved results: cnn_bilstm_architecture_comparison.csv")


Architecture Comparison:
                    Model  cnn_ch  lstm_h  lstm_layers  dropout  accuracy  \
0  CNN-BiLSTM-Attention A     128      64            2      0.6  0.541850   
1  CNN-BiLSTM-Attention B      64     128            1      0.5  0.506608   

   weighted_f1  trainable_params  
0     0.540135            402052  
1     0.500788            313156  

Best architecture: CNN-BiLSTM-Attention A
Best Accuracy: 54.19%
Saved model: cnn_bilstm_seed_iv_best_architecture.pth
Saved results: cnn_bilstm_architecture_comparison.csv
